In [ ]:
import pyspark
from pyspark import SparkContext
import imageio
import os
import numpy as np
import time
import matplotlib.pyplot as plt

In [ ]:
def readImg(path):
    #Read an image from file and convert it to a numpy array.
    #This function handles the file I/O operations.

    img = imageio.imread(path)
    im = np.array(img, dtype='uint8')
    return im

def writeImg(path, buf):
    #Write a processed image array to a file.
    #This function saves the output image to disk.
    imageio.imwrite(path, buf)

def display_images(original, filtered, title1="Original Image", title2="Filtered Image"):
    #Display the original and filtered images side by side for comparison.
    #This helps in visual evaluation of the filter's performance.
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title(title1)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(filtered)
    plt.title(title2)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def part_median_filter(local_data):
    #Apply median filter to a portion of the image.
    #This function runs in parallel on different partitions of the image.
    #Parameters: local_data: A list containing [partition_id, start_row, end_row, image_data]
    #Returns:Tuple of (partition_id, filtered_part)
    # Extract parameters from input data
    part_id = int(local_data[0])
    first = int(local_data[1])
    end = int(local_data[2])
    buf = local_data[3]
    
    # Get image dimensions
    nx = buf.shape[0]
    ny = buf.shape[1]
    
    # Determine number of color channels (handle both grayscale and color images)
    if len(buf.shape) > 2:
        nz = buf.shape[2]
        new_buf = np.zeros((end - first, ny, nz), dtype='uint8')
        is_color = True
    else:
        new_buf = np.zeros((end - first, ny), dtype='uint8')
        is_color = False
    
    # Print information about the current partition for debugging
    print(f"Processing partition {part_id}: rows {first} to {end}")
    
    # Apply median filter to each pixel
    for i in range(first, end):
        for j in range(ny):
            # For color images, process each channel separately
            if is_color:
                for c in range(nz):
                    # Collect the 3x3 neighborhood for median calculation
                    neighbors = []
                    for di in range(-1, 2):
                        for dj in range(-1, 2):
                            ni = min(max(i + di, 0), nx - 1)  # Ensure within image boundaries
                            nj = min(max(j + dj, 0), ny - 1)
                            neighbors.append(buf[ni, nj, c])
                    
                    # Replace pixel with median value of its neighborhood
                    new_buf[i - first, j, c] = np.median(neighbors)
            else:
                # Handle grayscale images
                neighbors = []
                for di in range(-1, 2):
                    for dj in range(-1, 2):
                        ni = min(max(i + di, 0), nx - 1)
                        nj = min(max(j + dj, 0), ny - 1)
                        neighbors.append(buf[ni, nj])
                
                new_buf[i - first, j] = np.median(neighbors)
    
    return part_id, new_buf

In [ ]:
def main():
    #Main function to orchestrate the parallel median filter process.
    #Reads the image, splits it into partitions, applies the filter in parallel,
    #and assembles the final filtered image.

    # Set the path to the input image
    data_dir = '.'  # Current directory, adjust as needed
    file = os.path.join(data_dir, 'lena_noisy.jpg')
    
    # Read the image
    img_buf = readImg(file)
    print('Image shape:', img_buf.shape)
    
    # Get image dimensions
    if len(img_buf.shape) > 2:
        nx, ny, nz = img_buf.shape
    else:
        nx, ny = img_buf.shape
        nz = 1
    
    # Split the image into partitions for parallel processing
    nb_partitions = 8  # Number of partitions
    print(f"Splitting image into {nb_partitions} partitions")
    
    # Prepare data for parallel processing
    data = []
    begin = 0
    block_size = nx // nb_partitions  # Integer division to ensure even splits
    
    for ip in range(nb_partitions):
        end = min(begin + block_size, nx)
        data.append([ip, begin, end, img_buf])
        begin = end
    
    # Create SparkContext and parallelize the data
    print("Initializing Spark context...")
    sc = SparkContext()
    data_rdd = sc.parallelize(data, nb_partitions)
    
    # Apply median filter in parallel
    print("Starting parallel median filter computation...")
    start_time = time.time()
    result_rdd = data_rdd.map(part_median_filter)
    result_data = result_rdd.collect()
    end_time = time.time()
    print(f"Parallel processing completed in {end_time - start_time:.4f} seconds")
    
    # Initialize the output image buffer
    if len(img_buf.shape) > 2:
        new_img_buf = np.zeros((nx, ny, nz), dtype='uint8')
    else:
        new_img_buf = np.zeros((nx, ny), dtype='uint8')
    
    # Assemble the final image from processed partitions
    for part_id, part_buf in sorted(result_data, key=lambda x: x[0]):
        start_row = data[part_id][1]
        end_row = data[part_id][2]
        if len(img_buf.shape) > 2:
            new_img_buf[start_row:end_row, :, :] = part_buf
        else:
            new_img_buf[start_row:end_row, :] = part_buf
    
    # Save the filtered image
    print('Creating filtered image file...')
    filter_file = os.path.join(data_dir, 'lena_filter.jpg')
    writeImg(filter_file, new_img_buf)
    print(f"Filtered image saved as {filter_file}")
    
    # Stop the Spark context
    sc.stop()
    
    # Return both original and filtered images for display
    return img_buf, new_img_buf

# Execute the main function
if __name__ == "__main__":
    original_img, filtered_img = main()

In [ ]:
# Display the original and filtered images side by side
original_img, filtered_img = main()
display_images(original_img, filtered_img, "Original Noisy Image", "Median Filtered Image")

# Calculate noise reduction metrics
def calculate_noise_reduction(original, filtered):
    """Calculate standard deviation reduction as a measure of noise removal"""
    original_std = np.std(original)
    filtered_std = np.std(filtered)
    reduction_percent = ((original_std - filtered_std) / original_std) * 100
    return original_std, filtered_std, reduction_percent

if len(original_img.shape) > 2:
    # For color images, calculate metrics per channel
    channels = ['Red', 'Green', 'Blue']
    for i in range(3):
        orig_std, filt_std, reduction = calculate_noise_reduction(
            original_img[:,:,i], filtered_img[:,:,i])
        print(f"{channels[i]} channel - Original std: {orig_std:.2f}, "
              f"Filtered std: {filt_std:.2f}, Noise reduction: {reduction:.2f}%")
else:
    # For grayscale images
    orig_std, filt_std, reduction = calculate_noise_reduction(original_img, filtered_img)
    print(f"Original std: {orig_std:.2f}, Filtered std: {filt_std:.2f}, "
          f"Noise reduction: {reduction:.2f}%")